# WariGuard — Moteur Final de Fusion IA

## Objectif

Ce notebook fusionne les trois briques de ta partie IA :

1. **Modèle binaire** : `arnaque` ou `legitime` ;
2. **Modèle multiclasse** : type d'arnaque `T1` à `T6` ;
3. **Moteur de règles V0** : mots déclencheurs et explicabilité.

La fonction finale à fournir à Elvis est :

```python
analyser_message_wariguard(texte)
```

Elle retourne un JSON directement exploitable dans le MVP.

# 1. Imports

In [1]:
import os
import re
import json
import joblib
import unicodedata
from typing import Dict, List

# 2. Chemins des modèles

Place ces fichiers dans le dossier `PROJET CLAPAY` :

- `wariguard_binary_model_augmented_hard.pkl`
- `wariguard_multiclass_model_augmented_hard.pkl`
- `wariguard_scenario_mapping.json`

In [2]:
PROJECT_DIR = "C:/Users/DELL/Desktop/PROJET/PROJET CLAPAY"

BINARY_MODEL_PATH = os.path.join(PROJECT_DIR, "wariguard_binary_model_augmented_hard.pkl")
MULTICLASS_MODEL_PATH = os.path.join(PROJECT_DIR, "wariguard_multiclass_model_augmented_hard.pkl")
MAPPING_PATH = os.path.join(PROJECT_DIR, "wariguard_scenario_mapping.json")

# Fallback local si le notebook est dans le même dossier que les fichiers.
if not os.path.exists(BINARY_MODEL_PATH):
    BINARY_MODEL_PATH = "wariguard_binary_model_augmented_hard.pkl"
if not os.path.exists(MULTICLASS_MODEL_PATH):
    MULTICLASS_MODEL_PATH = "wariguard_multiclass_model_augmented_hard.pkl"
if not os.path.exists(MAPPING_PATH):
    MAPPING_PATH = "wariguard_scenario_mapping.json"

print("Modèle binaire      :", BINARY_MODEL_PATH)
print("Modèle multiclasse :", MULTICLASS_MODEL_PATH)
print("Mapping scénarios  :", MAPPING_PATH)

if not os.path.exists(BINARY_MODEL_PATH):
    raise FileNotFoundError(f"Modèle binaire introuvable : {BINARY_MODEL_PATH}")
if not os.path.exists(MULTICLASS_MODEL_PATH):
    raise FileNotFoundError(f"Modèle multiclasse introuvable : {MULTICLASS_MODEL_PATH}")
if not os.path.exists(MAPPING_PATH):
    raise FileNotFoundError(f"Mapping introuvable : {MAPPING_PATH}")

Modèle binaire      : C:/Users/DELL/Desktop/PROJET/PROJET CLAPAY\wariguard_binary_model_augmented_hard.pkl
Modèle multiclasse : C:/Users/DELL/Desktop/PROJET/PROJET CLAPAY\wariguard_multiclass_model_augmented_hard.pkl
Mapping scénarios  : C:/Users/DELL/Desktop/PROJET/PROJET CLAPAY\wariguard_scenario_mapping.json


# 3. Chargement des modèles

In [3]:
binary_model = joblib.load(BINARY_MODEL_PATH)
multiclass_model = joblib.load(MULTICLASS_MODEL_PATH)

with open(MAPPING_PATH, "r", encoding="utf-8") as f:
    SCENARIO_MAPPING = json.load(f)

print("Modèles chargés avec succès.")
print("Classes binaires    :", binary_model.classes_)
print("Classes multiclasse :", multiclass_model.classes_)
print("Mapping             :", SCENARIO_MAPPING)

Modèles chargés avec succès.
Classes binaires    : ['arnaque' 'legitime']
Classes multiclasse : ['T1' 'T2' 'T3' 'T4' 'T5' 'T6']
Mapping             : {'T1': 'Faux agent Mobile Money', 'T2': 'Faux transfert erroné', 'T3': 'Faux gain / loterie', 'T4': 'Phishing / faux lien officiel', 'T5': "Usurpation d'urgence familiale", 'T6': 'Fausse promotion opérateur'}


# 4. Normalisation et règles explicables

Ces fonctions servent à extraire les mots déclencheurs et à renforcer légèrement le score final.

In [4]:
def normaliser_texte(texte: str) -> str:
    if not isinstance(texte, str):
        return ""
    texte = texte.lower()
    texte = unicodedata.normalize("NFD", texte)
    texte = "".join(c for c in texte if unicodedata.category(c) != "Mn")
    texte = re.sub(r"[^a-z0-9\s./:-]", " ", texte)
    texte = re.sub(r"\s+", " ", texte).strip()
    return texte

REGLES_MOTS_CLES = {
    "faux_agent": ["agent", "service client", "support", "operateur", "orange money", "wave", "mtn", "moov", "compte", "verification", "bloque", "otp", "pin", "code secret", "code recu", "message recu", "lisez-moi", "sequence recue", "numero temporaire"],
    "faux_gain": ["gagne", "gagnant", "felicitation", "promotion", "loterie", "bonus", "prix", "recompense", "gain", "dotation", "beneficiaire", "frais administratifs", "programme fidelite"],
    "faux_transfert": ["transfert", "erreur", "mauvais transfert", "renvoyer", "rembourser", "restituer", "retourner", "confusion", "pas destine"],
    "phishing_lien": ["connectez-vous", "reconnectez", "validez", "confirmez", "lien", "http", "https", "restriction", "acces limite", "page de verification"],
    "urgence_familiale": ["accident", "hopital", "urgence", "commissariat", "police", "telephone d un ami", "portefeuille disparu", "probleme", "maman", "papa", "tantie"],
    "fausse_promotion": ["double", "triple", "retour automatique", "multiplication", "offre speciale", "operation speciale", "solde multiplie", "depot minimum"],
    "nouchi_mixte": ["mon vieux", "chef", "tantie", "yako", "compte-la", "affaire-la", "faut pas dormir", "drap"]
}

SIGNAUX_CRITIQUES = ["otp", "pin", "code secret", "code confidentiel", "mot de passe", "code recu", "message recu", "lisez-moi", "sequence recue", "http", "https"]

def extraire_mots_declencheurs(texte: str) -> List[str]:
    texte_norm = normaliser_texte(texte)
    mots = []
    for categorie, liste_mots in REGLES_MOTS_CLES.items():
        for mot in liste_mots:
            mot_norm = normaliser_texte(mot)
            if mot_norm and mot_norm in texte_norm:
                mots.append(mot)
    montants = re.findall(r"\b\d{3,}\s*(?:f|fcfa)?\b", texte_norm)
    mots.extend([m.strip() for m in montants])
    return sorted(set(mots))

def calculer_bonus_regles(texte: str, mots_declencheurs: List[str]) -> float:
    texte_norm = normaliser_texte(texte)
    bonus = 0.0
    for signal in SIGNAUX_CRITIQUES:
        if normaliser_texte(signal) in texte_norm:
            bonus += 0.05
    if len(mots_declencheurs) >= 3:
        bonus += 0.05
    if len(mots_declencheurs) >= 6:
        bonus += 0.05
    return min(bonus, 0.20)

# 5. Message utilisateur

In [5]:
def generer_message_utilisateur(resultat: Dict) -> str:
    niveau = resultat.get("niveau_alerte", "vert")
    score = int(resultat.get("score_risque", 0) * 100)
    type_arnaque = resultat.get("type_arnaque", "aucune")

    if niveau == "rouge":
        return (
            f"🔴 Alerte élevée ({score}%)\n"
            f"Cette communication présente de forts signes d'arnaque.\n"
            f"Type détecté : {type_arnaque}.\n"
            f"Conseil : ne communiquez aucun code et ne transférez pas d'argent."
        )
    elif niveau == "orange":
        return (
            f"🟠 Prudence ({score}%)\n"
            f"Des éléments suspects ont été détectés.\n"
            f"Type probable : {type_arnaque}.\n"
            f"Conseil : vérifiez l'identité de votre interlocuteur avant toute action."
        )
    else:
        return (
            f"🟢 Aucun risque détecté ({score}%)\n"
            f"Ce message ne présente pas de signe évident d'arnaque connu."
        )

# 6. Fonction finale : `analyser_message_wariguard`

Cette fonction est le contrat final pour l'application.

In [6]:
def analyser_message_wariguard(texte: str) -> Dict:
    # Modèle binaire
    label_pred = binary_model.predict([texte])[0]
    binary_proba = binary_model.predict_proba([texte])[0]
    binary_proba_dict = dict(zip(binary_model.classes_, binary_proba))
    score_ml = float(binary_proba_dict.get("arnaque", 0.0))

    # Règles explicables
    mots_declencheurs = extraire_mots_declencheurs(texte)
    bonus_regles = calculer_bonus_regles(texte, mots_declencheurs)

    # Score final borné entre 0 et 1
    score_final = min(score_ml + bonus_regles, 1.0)

    if score_final >= 0.70:
        niveau_alerte = "rouge"
    elif score_final >= 0.40:
        niveau_alerte = "orange"
    else:
        niveau_alerte = "vert"

    # Si faible risque, pas besoin de modèle multiclasse
    if score_final < 0.40:
        resultat = {
            "score_risque": round(score_final, 4),
            "niveau_alerte": niveau_alerte,
            "label": "legitime",
            "type_scenario": "T7",
            "type_arnaque": "aucune",
            "mots_declencheurs": mots_declencheurs,
            "probabilite_ml_arnaque": round(score_ml, 4),
            "bonus_regles": round(bonus_regles, 4),
            "probabilites_binaires": {k: round(float(v), 4) for k, v in binary_proba_dict.items()},
            "probabilites_types": {}
        }
        resultat["message_utilisateur"] = generer_message_utilisateur(resultat)
        return resultat

    # Si risque moyen ou élevé, prédire le type d'arnaque
    type_pred = multiclass_model.predict([texte])[0]
    type_proba = multiclass_model.predict_proba([texte])[0]
    type_proba_dict = dict(zip(multiclass_model.classes_, type_proba))

    resultat = {
        "score_risque": round(score_final, 4),
        "niveau_alerte": niveau_alerte,
        "label": "arnaque",
        "type_scenario": type_pred,
        "type_arnaque": SCENARIO_MAPPING.get(type_pred, type_pred),
        "confiance_type": round(float(max(type_proba)), 4),
        "mots_declencheurs": mots_declencheurs,
        "probabilite_ml_arnaque": round(score_ml, 4),
        "bonus_regles": round(bonus_regles, 4),
        "probabilites_binaires": {k: round(float(v), 4) for k, v in binary_proba_dict.items()},
        "probabilites_types": {SCENARIO_MAPPING.get(k, k): round(float(v), 4) for k, v in type_proba_dict.items()}
    }
    resultat["message_utilisateur"] = generer_message_utilisateur(resultat)
    return resultat

# 7. Tests du moteur final

In [7]:
tests = [
    "Bonjour je suis un agent Orange Money, votre compte est bloque, lisez-moi le code recu par SMS.",
    "Message Orange Money : ne communiquez jamais votre code secret a qui que ce soit.",
    "Votre numero a ete retenu pour une dotation client, reglez les frais administratifs pour finaliser le retrait.",
    "Bonsoir, j'ai envoye 25000F par erreur sur votre numero, merci de me restituer le montant rapidement.",
    "Suite a une verification de routine, reconnectez votre compte Wave ici : https://wave-ci-secure.test/verif",
    "Maman je suis bloque a la gare, portefeuille disparu, envoie quelque chose vite sur ce numero.",
    "Faut pas dormir, Moov Money donne retour double aux premiers clients qui valident transaction-la maintenant.",
    "Bonjour, la reunion du projet est reportee a demain matin.",
    "Transaction confirmee : transfert de 25000F effectue avec succes. Ne partagez jamais votre PIN."
]

for texte in tests:
    resultat = analyser_message_wariguard(texte)
    print("\n" + "=" * 100)
    print("TEXTE :", texte)
    print("\nJSON TECHNIQUE :")
    print(json.dumps(resultat, ensure_ascii=False, indent=2))
    print("\nMESSAGE UTILISATEUR :")
    print(resultat["message_utilisateur"])


TEXTE : Bonjour je suis un agent Orange Money, votre compte est bloque, lisez-moi le code recu par SMS.

JSON TECHNIQUE :
{
  "score_risque": 1.0,
  "niveau_alerte": "rouge",
  "label": "arnaque",
  "type_scenario": "T1",
  "type_arnaque": "Faux agent Mobile Money",
  "confiance_type": 0.7609,
  "mots_declencheurs": [
    "agent",
    "bloque",
    "code recu",
    "compte",
    "lisez-moi",
    "orange money"
  ],
  "probabilite_ml_arnaque": 0.8955,
  "bonus_regles": 0.2,
  "probabilites_binaires": {
    "arnaque": 0.8955,
    "legitime": 0.1045
  },
  "probabilites_types": {
    "Faux agent Mobile Money": 0.7609,
    "Faux transfert erroné": 0.0675,
    "Faux gain / loterie": 0.0359,
    "Phishing / faux lien officiel": 0.0455,
    "Usurpation d'urgence familiale": 0.0548,
    "Fausse promotion opérateur": 0.0355
  },
  "message_utilisateur": "🔴 Alerte élevée (100%)\nCette communication présente de forts signes d'arnaque.\nType détecté : Faux agent Mobile Money.\nConseil : ne commun

# 8. Sauvegarde du contrat JSON pour Elvis

Cette cellule produit un exemple de sortie JSON que l'équipe front/MVP peut utiliser.

In [8]:
exemple_contrat = analyser_message_wariguard(
    "Je suis agent Wave, lis-moi le message recu pour confirmer ton compte maintenant."
)

CONTRACT_PATH = "wariguard_api_contract_example.json"
with open(CONTRACT_PATH, "w", encoding="utf-8") as f:
    json.dump(exemple_contrat, f, ensure_ascii=False, indent=2)

print("Contrat JSON sauvegardé :", CONTRACT_PATH)
print(json.dumps(exemple_contrat, ensure_ascii=False, indent=2))

Contrat JSON sauvegardé : wariguard_api_contract_example.json
{
  "score_risque": 0.9021,
  "niveau_alerte": "rouge",
  "label": "arnaque",
  "type_scenario": "T1",
  "type_arnaque": "Faux agent Mobile Money",
  "confiance_type": 0.579,
  "mots_declencheurs": [
    "agent",
    "compte",
    "message recu",
    "wave"
  ],
  "probabilite_ml_arnaque": 0.8021,
  "bonus_regles": 0.1,
  "probabilites_binaires": {
    "arnaque": 0.8021,
    "legitime": 0.1979
  },
  "probabilites_types": {
    "Faux agent Mobile Money": 0.579,
    "Faux transfert erroné": 0.0852,
    "Faux gain / loterie": 0.0929,
    "Phishing / faux lien officiel": 0.0616,
    "Usurpation d'urgence familiale": 0.1168,
    "Fausse promotion opérateur": 0.0644
  },
  "message_utilisateur": "🔴 Alerte élevée (90%)\nCette communication présente de forts signes d'arnaque.\nType détecté : Faux agent Mobile Money.\nConseil : ne communiquez aucun code et ne transférez pas d'argent."
}


# 9. Conclusion

Ce notebook produit la brique finale de ta partie IA :

```text
Texte entrant
        ↓
Modèle binaire
        ↓
Modèle multiclasse si arnaque
        ↓
Moteur de règles explicable
        ↓
JSON final pour l'application
```

## Fichier généré

- `wariguard_api_contract_example.json`

## À transmettre à Elvis

- `wariguard_binary_model_augmented_hard.pkl`
- `wariguard_multiclass_model_augmented_hard.pkl`
- `wariguard_scenario_mapping.json`
- ce notebook ou la fonction `analyser_message_wariguard`
- `wariguard_api_contract_example.json`